# Notebook 04 — Mod30 SAE Toy Model

**Purpose:** move from exact residue detectors to a toy sparse-recovery experiment.

Notebook 01: finite Mod30 residue manifold.  
Notebook 02: global detector vs local tile union.  
Notebook 03: sparse-feature representation metrics.  
Notebook 04: continuous embeddings + sparse factorization recovery.

Core claim:

```text
When residue lanes are embedded as continuous signals,
sparse recovery shifts from global blur toward local Mod30 tile recovery
as feature count increases.
```

Model choice:

```text
Nonnegative Matrix Factorization (NMF)
```

NMF is not a full SAE, but it gives a lightweight sparse-feature analogue:

```text
continuous observations X ≈ activations W × features H
```


## 0. Bulletproof setup

Run first.

In [ ]:

from pathlib import Path
import sys

def find_repo_root(start=None, marker="src"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""from math import gcd
MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
def mod_index(n, mod): return n % mod
def mod_mask(n, residues, mod): return mod_index(n, mod) in residues
def generate_coprime_residues(mod): return [r for r in range(1, mod) if gcd(r, mod) == 1]
def mod30_index(n): return mod_index(n, MOD30)
def mod30_mask(n): return mod_mask(n, MOD30_RESIDUES, MOD30)
def residue_to_lane_index(residue): return MOD30_RESIDUES.index(residue) if residue in MOD30_RESIDUES else -1
def mod30_residues(n_max): return [n for n in range(2, n_max) if mod30_mask(n)]
def single_lane_mask(n, lane=1): return mod30_index(n) == lane
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""import numpy as np
def lane_coverage(captured_residues, target_residues):
    captured, target = set(captured_residues), set(target_residues)
    hit, missed = captured & target, target - captured
    return {"target_lanes": len(target), "captured_lanes": len(hit), "missed_lanes": len(missed),
            "coverage_fraction": len(hit)/len(target) if target else 0.0,
            "captured_residues": sorted(hit), "missed_residues": sorted(missed)}
def reconstruction_error(X, X_hat):
    denom = np.linalg.norm(X)
    return 0.0 if denom == 0 else float(np.linalg.norm(X - X_hat) / denom)
def activation_sparsity(A, eps=1e-6): return float(np.mean(A <= eps))
def feature_lane_alignment(A, lane_labels, residues):
    align = np.zeros((A.shape[1], len(residues)))
    for j in range(A.shape[1]):
        total = A[:, j].sum()
        if total <= 0: continue
        for k, r in enumerate(residues):
            mask = lane_labels == r
            align[j, k] = A[mask, j].sum() / total
    return align
def lane_purity_from_alignment(align): return [] if align.size == 0 else align.max(axis=1).tolist()
def recovered_residues_from_alignment(align, residues, threshold=0.45):
    recovered = []
    for row in align:
        if row.max() >= threshold:
            recovered.append(residues[int(row.argmax())])
    return sorted(set(recovered))
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""import matplotlib.pyplot as plt
def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler

from src.mod30 import MOD30_RESIDUES, mod30_index, mod30_mask, residue_to_lane_index
from src.tiling_metrics import (
    lane_coverage,
    reconstruction_error,
    activation_sparsity,
    feature_lane_alignment,
    lane_purity_from_alignment,
    recovered_residues_from_alignment,
)
from src.plots import save_current

RNG = np.random.default_rng(9423)
print("Persisting Mod30 lanes:", MOD30_RESIDUES)


## 1. Build Mod30 dataset

We use several periods so the sparse model sees repeated local structure.


In [ ]:
n_min = 1
n_max = 1500
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))
df["lane_index"] = df["mod30_residue"].apply(residue_to_lane_index)

gated_df = df[df["inside_mod30_gate"]].copy().reset_index(drop=True)
gated_df.head()


## 2. Create synthetic continuous embeddings

Each gated integer receives a continuous vector.

Embedding components:

```text
1. circular residue coordinates: cos(theta), sin(theta)
2. 8 lane-specific channels
3. small noise channels
```

This converts exact Mod30 residue lanes into continuous observations.


In [ ]:
def build_continuous_embedding(gated_df, noise_scale=0.04, lane_strength=1.0):
    residues = gated_df["mod30_residue"].to_numpy()
    lane_idx = gated_df["lane_index"].to_numpy()

    theta = 2 * np.pi * residues / 30.0

    # Shift to nonnegative for NMF compatibility.
    circle = np.column_stack([
        0.5 + 0.5 * np.cos(theta),
        0.5 + 0.5 * np.sin(theta),
    ])

    lane_channels = np.zeros((len(gated_df), len(MOD30_RESIDUES)))
    lane_channels[np.arange(len(gated_df)), lane_idx] = lane_strength

    # Add smooth local overlap so lanes are not perfectly one-hot.
    # This makes recovery more SAE-like: local neighborhoods blur.
    for i in range(len(gated_df)):
        k = lane_idx[i]
        lane_channels[i, (k - 1) % len(MOD30_RESIDUES)] += 0.18
        lane_channels[i, (k + 1) % len(MOD30_RESIDUES)] += 0.18

    noise = RNG.normal(loc=0.0, scale=noise_scale, size=(len(gated_df), 4))
    noise = np.abs(noise)

    X = np.column_stack([circle, lane_channels, noise])
    X = MinMaxScaler().fit_transform(X)
    return X

X = build_continuous_embedding(gated_df)
X.shape


## 3. Visualize synthetic residue circle

The model observes continuous vectors, but we retain known lane labels for evaluation.


In [ ]:
circle_x = X[:, 0]
circle_y = X[:, 1]

plt.figure(figsize=(6, 6))
plt.scatter(circle_x, circle_y, c=gated_df["lane_index"], s=10)
plt.xlabel("embedding dim 0")
plt.ylabel("embedding dim 1")
plt.title("Synthetic continuous embedding: Mod30 residue circle")
save_current(FIGURES_DIR / "16_synthetic_embedding_residue_circle.png")
plt.show()


## 4. Ground-truth tile matrix

This is the target structure: eight persisting Mod30 residue lanes.


In [ ]:
Y_true = np.zeros((len(gated_df), len(MOD30_RESIDUES)))
Y_true[np.arange(len(gated_df)), gated_df["lane_index"].to_numpy()] = 1

plt.figure(figsize=(12, 4.5))
plt.imshow(Y_true[:240].T, aspect="auto", interpolation="nearest")
plt.yticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
plt.xlabel("sample index")
plt.ylabel("true lane")
plt.title("Ground-truth Mod30 tile matrix")
save_current(FIGURES_DIR / "17_ground_truth_tile_matrix.png")
plt.show()


## 5. Fit NMF models over feature counts

We test:

```text
1 component  → global blur
4 components → grouped tiles
8 components → local tile recovery
12 components → overcomplete / redundant recovery
```


In [ ]:
component_grid = [1, 4, 8, 12]
models = {}
metrics = []
alignment_mats = {}

for k in component_grid:
    nmf = NMF(
        n_components=k,
        init="nndsvda",
        random_state=9423,
        max_iter=1500,
        l1_ratio=0.65,
        alpha_W=0.002,
        alpha_H=0.002,
    )
    W = nmf.fit_transform(X)
    H = nmf.components_
    X_hat = W @ H

    align = feature_lane_alignment(W, gated_df["mod30_residue"].to_numpy(), MOD30_RESIDUES)
    purities = lane_purity_from_alignment(align)
    recovered = recovered_residues_from_alignment(align, MOD30_RESIDUES, threshold=0.45)
    cov = lane_coverage(recovered, MOD30_RESIDUES)

    models[k] = {"model": nmf, "W": W, "H": H, "X_hat": X_hat}
    alignment_mats[k] = align

    total_activations = float((W > 1e-6).sum())
    redundancy = total_activations / cov["captured_lanes"] if cov["captured_lanes"] else np.inf

    metrics.append({
        "n_components": k,
        "reconstruction_error": reconstruction_error(X, X_hat),
        "activation_sparsity": activation_sparsity(W, eps=1e-6),
        "mean_lane_purity": float(np.mean(purities)) if purities else 0.0,
        "max_lane_purity": float(np.max(purities)) if purities else 0.0,
        "captured_lanes": cov["captured_lanes"],
        "coverage_fraction": cov["coverage_fraction"],
        "total_activations": total_activations,
        "redundancy": redundancy,
        "recovered_residues": recovered,
        "missed_residues": cov["missed_residues"],
    })

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(DATA_DIR / "04_nmf_recovery_metrics.csv", index=False)
metrics_df


## 6. Learned activation matrix

Show the learned NMF activations for the strongest direct tile setting: `n_components = 8`.


In [ ]:
k_show = 8
W_show = models[k_show]["W"]

plt.figure(figsize=(12, 4.8))
plt.imshow(W_show[:240].T, aspect="auto", interpolation="nearest")
plt.xlabel("sample index")
plt.ylabel("learned component")
plt.title("NMF learned activation matrix: 8 components")
save_current(FIGURES_DIR / "18_nmf_learned_activation_matrix.png")
plt.show()


## 7. Feature-to-lane alignment matrix

Rows are learned features.  
Columns are true Mod30 lanes.

A strong diagonal or near-diagonal pattern means learned features align with local residue tiles.


In [ ]:
align_show = alignment_mats[k_show]

plt.figure(figsize=(8, 5))
plt.imshow(align_show, aspect="auto", interpolation="nearest")
plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
plt.yticks(range(k_show), [f"F{j}" for j in range(k_show)])
plt.xlabel("true residue lane")
plt.ylabel("learned component")
plt.title("Feature-to-lane alignment: 8-component NMF")
plt.colorbar(label="fraction of component activation")
save_current(FIGURES_DIR / "19_feature_to_lane_alignment.png")
plt.show()


## 8. Recovery metrics summary

This plot summarizes feature-count effects:

```text
more components → better tile coverage
too many components → possible redundancy
```


In [ ]:
x = metrics_df["n_components"].astype(str)
width = 0.2
idx = np.arange(len(metrics_df))

plt.figure(figsize=(10, 5.5))
plt.bar(idx - 1.5*width, metrics_df["coverage_fraction"], width, label="coverage")
plt.bar(idx - 0.5*width, metrics_df["mean_lane_purity"], width, label="mean purity")
plt.bar(idx + 0.5*width, 1 - metrics_df["reconstruction_error"], width, label="1 - recon error")
plt.bar(idx + 1.5*width, metrics_df["activation_sparsity"], width, label="activation sparsity")

plt.xticks(idx, x)
plt.xlabel("NMF component count")
plt.ylabel("metric value")
plt.title("NMF recovery metrics: global blur → local tile recovery")
plt.legend()
save_current(FIGURES_DIR / "20_recovery_metrics_summary.png")
plt.show()


## 9. Reconstruction error vs feature count

This is the model-side analogue of compression vs local recovery.


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(metrics_df["n_components"], metrics_df["reconstruction_error"], marker="o")
plt.xlabel("NMF component count")
plt.ylabel("relative reconstruction error")
plt.title("Reconstruction error vs feature count")
save_current(FIGURES_DIR / "21_reconstruction_error_vs_feature_count.png")
plt.show()


## 10. Alignment matrices across feature counts

This checks whether low component counts compress lanes together and higher counts recover local tiles.


In [ ]:
for k in component_grid:
    align = alignment_mats[k]
    plt.figure(figsize=(8, max(2.5, 0.45 * k)))
    plt.imshow(align, aspect="auto", interpolation="nearest")
    plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
    plt.yticks(range(k), [f"F{j}" for j in range(k)])
    plt.xlabel("true residue lane")
    plt.ylabel("learned component")
    plt.title(f"Feature-to-lane alignment: {k} components")
    plt.colorbar(label="fraction of component activation")
    save_current(FIGURES_DIR / f"22_alignment_matrix_{k}_components.png")
    plt.show()


## 11. Interpretation

Observed pattern to look for:

| Components | Expected representation |
|---:|---|
| 1 | global blur |
| 4 | grouped local tiles |
| 8 | near-local Mod30 tile recovery |
| 12 | overcomplete / redundant tiling |

Paper-facing phrase:

```text
As feature count increases, learned sparse components shift from global blur toward local Mod30 tile recovery.
```

This is the closest notebook so far to the SAE paper’s modeling situation:

```text
known latent structure
continuous observations
sparse learned components
measurable tile recovery
```


## 12. Save compact summary

In [ ]:
summary_md = f"""# Notebook 04 Summary — Mod30 SAE Toy Model

Notebook 04 converts exact Mod30 residue lanes into continuous synthetic embeddings
and tests whether a lightweight sparse factorization recovers local tile structure.

## Model

Nonnegative Matrix Factorization:

`X ≈ W H`

## Component grid

`{component_grid}`

## Metrics

{metrics_df.to_markdown(index=False)}

## Interpretation

As feature count increases, learned sparse components shift from global blur toward local Mod30 tile recovery.

## Generated figures

- `figures/16_synthetic_embedding_residue_circle.png`
- `figures/17_ground_truth_tile_matrix.png`
- `figures/18_nmf_learned_activation_matrix.png`
- `figures/19_feature_to_lane_alignment.png`
- `figures/20_recovery_metrics_summary.png`
- `figures/21_reconstruction_error_vs_feature_count.png`
- `figures/22_alignment_matrix_{{k}}_components.png` for k in `{component_grid}`

## Generated data

- `data/04_nmf_recovery_metrics.csv`
"""

summary_path = OUTPUTS_DIR / "04_mod30_sae_toy_model_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 13. Optional: zip-download pattern

Uncomment in Colab to download figures, data, and output summaries.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_04_mod30_sae_toy_model_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 14. Recommended next step

At this point the repo has enough notebook evidence for a short paper draft.

Recommended next file:

```text
paper/mod30_manifold_tiling.tex
```

Working title:

```text
Residue-Class Tiling as a Finite Analogue for Sparse Feature Capture
```
